# Phase 5 — 실습 문제 정답

정답 코드와 간단한 설명입니다.

## 0. 그래프 한글 폰트 설정

그래프 라벨에 한글을 쓸 때 깨지지 않도록, 설치된 한글 폰트를 찾아 적용합니다. (없으면 라벨은 영어로 표기)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; print('한글 폰트 적용:', _n); break
else:
    print('한글 폰트를 찾지 못했습니다 — 라벨은 영어로 표기합니다.')
plt.rcParams['axes.unicode_minus'] = False

## 1. 예제 이미지 준비 (먼저 실행)
아래 셀을 실행해 실습용 이미지 `img` 를 준비하십시오.

In [ ]:
import numpy as np, cv2
np.random.seed(7)
H,W=240,360
base=np.zeros((H,W)); base[:70]=95; base[70:100]=150; base[100:120]=60; base[120:175]=135; base[175:]=100
base=base*0.55+70
img=np.clip(base+np.random.normal(0,12,(H,W)),0,255).astype(np.uint8)
print('img 준비 완료:', img.shape)

### 문제 1. 정규화
NORM_MINMAX 로 0~255 로 늘립니다.

In [ ]:
norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
norm = norm.astype(np.uint8)

assert norm.min() == 0 and norm.max() == 255
print('통과')

### 문제 2. 노이즈 제거 후 Otsu 이진화
GaussianBlur 후 THRESH_OTSU 로 임계값을 자동 결정합니다.

In [ ]:
den = cv2.GaussianBlur(img, (5,5), 0)
thr, binary = cv2.threshold(den, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

assert set(np.unique(binary)).issubset({0, 255})
assert 0 < thr < 255
print('통과')

### 문제 3. 모폴로지 열림
MORPH_OPEN 으로 작은 흰 점을 제거합니다.

In [ ]:
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)

assert opened.shape == binary.shape
assert set(np.unique(opened)).issubset({0, 255})
print('통과')

### 문제 4. 값 기반 마스크
inRange 로 밝기 구간을 선택하고 countNonZero 로 개수를 셉니다.

In [ ]:
mask = cv2.inRange(img, 140, 255)
cnt = cv2.countNonZero(mask)

assert mask.shape == img.shape
assert cnt == int((img >= 140).sum())
print('통과')

### 문제 5. 윤곽선 개수
findContours 의 반환값 중 첫 번째가 윤곽선 목록입니다.

In [ ]:
den = cv2.GaussianBlur(img,(5,5),0)
_, binary = cv2.threshold(den,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)
cnts, _ = cv2.findContours(opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
n = len(cnts)

assert n >= 1
print('통과 — 윤곽선', n, '개')

---
정답 확인을 마칩니다.